# 🎧 Sintonia — 2. Explorar o catálogo

Análise exploratória: o que tem no catálogo, como as características se distribuem e como as músicas se agrupam.

Rode depois do notebook 01. Funciona no Colab ou no seu computador.

In [ ]:
import os, sys, sqlite3
import pandas as pd
import plotly.express as px

# no Colab: baixa o projeto do GitHub (com o catálogo em dados/catalogo.db). Sem acesso ao Google Drive.
REPO = "https://github.com/laisedosdados/sintonia.git"
if "google.colab" in sys.modules:
    if not os.path.exists("/content/sintonia"):
        !git clone -q {REPO} /content/sintonia
    os.chdir("/content/sintonia")
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
DB = "dados/catalogo.db"
sys.path.insert(0, os.getcwd())
con = sqlite3.connect(DB)
print("Usando:", os.path.abspath(DB))

## 1. Visão geral (SQL)

In [ ]:
pd.read_sql("""
    SELECT fonte,
           COUNT(*)                        AS musicas,
           SUM(tem_voz)                    AS com_voz,
           ROUND(AVG(segundos_analisados)) AS segundos_medios
    FROM musicas
    GROUP BY fonte
""", con)

In [ ]:
# tons mais comuns
tons = pd.read_sql("SELECT tom, COUNT(*) AS musicas FROM musicas GROUP BY tom ORDER BY musicas DESC", con)
px.bar(tons, x="tom", y="musicas", title="Tons estimados no catálogo")

## 2. Distribuição das características
Escolha uma característica (veja os nomes em `sintonia/descricoes.py`).

In [ ]:
CARACTERISTICA = "rit_andamento"   # tente: mel_saltos, voz_vibrato, har_maior_menor, som_graves

dados = pd.read_sql("""
    SELECT m.titulo, m.artista, m.fonte, c.valor
    FROM caracteristicas c JOIN musicas m ON m.id = c.musica_id
    WHERE c.nome = ? AND c.valor IS NOT NULL
""", con, params=[CARACTERISTICA])
px.histogram(dados, x="valor", color="fonte", nbins=40, barmode="overlay",
             hover_data=["titulo", "artista"], title=CARACTERISTICA)

In [ ]:
# os extremos: quem tem mais e quem tem menos
pd.concat([dados.nlargest(5, "valor").assign(ponta="mais"),
           dados.nsmallest(5, "valor").assign(ponta="menos")])

## 3. O mapa musical
Cada ponto é uma música. Pontos próximos são parecidos **no aspecto escolhido**. Passe o mouse para ver os nomes.

In [ ]:
from sintonia.recomendador import Recomendador
from sintonia.exportar import mapa

rec = Recomendador.do_banco(DB)
mp = mapa(rec).merge(rec.musicas[["titulo", "artista", "fonte"]], left_on="musica_id", right_index=True)

ASPECTO = "melodia"   # geral, melodia, voz, ritmo, harmonia, som
px.scatter(mp[mp.aspecto == ASPECTO], x="x", y="y", color="fonte",
           hover_data=["titulo", "artista"], title=f"Mapa de {ASPECTO}", height=600)

## 4. Testar o recomendador sem o app
Escolha uma música do catálogo e pesos para cada aspecto.

In [ ]:
from sintonia.recomendador import achatar

BUSCA = "evid"   # parte do título
semente_id = rec.musicas[rec.musicas.titulo.str.lower().str.contains(BUSCA)].index[0]
print("Semente:", rec.musicas.loc[semente_id, "titulo"], "—", rec.musicas.loc[semente_id, "artista"])

linha = rec.X.loc[semente_id]
feats = {"todos": linha.to_dict()}
pesos = {"melodia": 0.75, "voz": 0.25}

res, avisos = rec.recomendar(feats, pesos, n=10, excluir=rec.musicas.loc[semente_id].to_dict() | {"id": semente_id})
res[["titulo", "artista", "parecida", "parecida_melodia", "parecida_voz", "explicacao"]]

## 5. Avaliar o recomendador
Quando pessoas usam o app e clicam 👍/👎, isso vai para `dados/uso.db`. Com isso dá para medir a **precisão**: das recomendações avaliadas, quantas acertaram?

In [ ]:
if os.path.exists("dados/uso.db"):
    uso = sqlite3.connect("dados/uso.db")
    display(pd.read_sql("""
        SELECT posicao, COUNT(gostou) AS avaliadas, ROUND(AVG(gostou) * 100, 1) AS acerto_pct
        FROM recomendacoes WHERE gostou IS NOT NULL
        GROUP BY posicao ORDER BY posicao
    """, uso))
else:
    print("Ainda não há dados de uso. Rode o app e avalie algumas recomendações!")